# 🎵 MoodBeats AI — Session 6: Music Recommendations & Results Page

Today we add **clickable Spotify and YouTube links** to every song — without any API keys.

---
## The Trick: Search URLs

Spotify and YouTube Music both have public search URLs:

```
https://open.spotify.com/search/happy%20pharrell
https://music.youtube.com/search?q=happy+pharrell
```

We just **construct these URLs** from the song title and artist. Zero API keys needed!

---
## Part 1: URL Encoding

URLs cannot contain spaces or special characters. We must **encode** them.

```
"Happy Pharrell" → "Happy%20Pharrell"  (or "Happy+Pharrell")
```

In [ ]:
from urllib.parse import quote, quote_plus

title  = "Don't Stop Me Now"
artist = "Queen"

query = f"{title} {artist}"
print(f"Raw query:        {query}")
print(f"quote():          {quote(query)}")
print(f"quote_plus():     {quote_plus(query)}")

### Building the search URLs

In [ ]:
def spotify_link(title: str, artist: str) -> str:
    query = quote(f"{title} {artist}")
    return f"https://open.spotify.com/search/{query}"

def yt_music_link(title: str, artist: str) -> str:
    query = quote_plus(f"{title} {artist}")
    return f"https://music.youtube.com/search?q={query}"

def youtube_link(title: str, artist: str) -> str:
    query = quote_plus(f"{title} {artist} official")
    return f"https://www.youtube.com/results?search_query={query}"

# Test with a real song
t, a = "Happy", "Pharrell Williams"
print(spotify_link(t, a))
print(yt_music_link(t, a))
print(youtube_link(t, a))

### ✏️ In-Class Exercise 6a — Test Links in Browser

Generate links for 3 songs and open them to verify they work.

In [ ]:
my_songs = [
    ("Blinding Lights", "The Weeknd"),
    # TODO: Add 2 more (title, artist) tuples
]

for title, artist in my_songs:
    print(f"\n{title} — {artist}")
    print(f"  Spotify:  {spotify_link(title, artist)}")
    print(f"  YT Music: {yt_music_link(title, artist)}")

---
## Part 2: The enrich() Function

`enrich()` adds music links to the raw Gemini result dict.

In [ ]:
def enrich(result: dict) -> dict:
    """Add spotify_url, yt_music_url, youtube_url to each song in-place."""
    for song in result.get("songs", []):
        t = song.get("title", "")
        a = song.get("artist", "")
        song["spotify_url"]  = spotify_link(t, a)
        song["yt_music_url"] = yt_music_link(t, a)
        song["youtube_url"]  = youtube_link(t, a)
    return result

# Test with a mock result
mock_result = {
    "emotion": "happy",
    "confidence": 0.92,
    "mood_description": "Feeling joyful and full of energy!",
    "songs": [
        {"title": "Happy",       "artist": "Pharrell Williams", "genre": "Pop"},
        {"title": "Good Feeling","artist": "Flo Rida",          "genre": "Hip-Hop"},
        {"title": "Uptown Funk", "artist": "Bruno Mars",        "genre": "Funk"},
    ]
}

enrich(mock_result)

# Check the enriched songs
for song in mock_result["songs"]:
    print(f"\n{song['title']}:")
    print(f"  Spotify: {song['spotify_url']}")

---
## Part 3: Emotion Themes

Each emotion gets its own colour scheme. We use **hex colours** with inline `style=` attributes  
because Tailwind CDN cannot process class names built at runtime.

In [ ]:
EMOTION_THEMES = {
    "happy":     {"accent": "#f59e0b", "bg": "#78350f", "border": "#d97706", "emoji": "😊"},
    "sad":       {"accent": "#60a5fa", "bg": "#1e3a5f", "border": "#3b82f6", "emoji": "😢"},
    "angry":     {"accent": "#f87171", "bg": "#7f1d1d", "border": "#ef4444", "emoji": "😤"},
    "surprised": {"accent": "#c084fc", "bg": "#4c1d95", "border": "#a855f7", "emoji": "😲"},
    "fearful":   {"accent": "#6ee7b7", "bg": "#064e3b", "border": "#10b981", "emoji": "😨"},
    "disgusted": {"accent": "#86efac", "bg": "#14532d", "border": "#22c55e", "emoji": "🤢"},
    "calm":      {"accent": "#67e8f9", "bg": "#164e63", "border": "#06b6d4", "emoji": "😌"},
    "excited":   {"accent": "#fb923c", "bg": "#7c2d12", "border": "#f97316", "emoji": "🤩"},
    "neutral":   {"accent": "#9ca3af", "bg": "#1f2937", "border": "#6b7280", "emoji": "😐"},
}

DEFAULT_THEME = {"accent": "#6366f1", "bg": "#1e1b4b", "border": "#4f46e5", "emoji": "🎵"}

def get_theme(emotion: str) -> dict:
    return EMOTION_THEMES.get(emotion.lower(), DEFAULT_THEME)

# Test
print(get_theme("happy"))
print(get_theme("unknown"))  # should return DEFAULT_THEME

---
## Part 4: The Results Template with Song Cards

In [ ]:
import os
os.makedirs("templates", exist_ok=True)

with open("templates/results_v2.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html><head>
<title>MoodBeats — Results</title>
<script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-950 text-white min-h-screen p-6">

<!-- Emotion Banner: uses inline style because Tailwind CDN can't JIT dynamic values -->
<div class="rounded-2xl p-8 mb-8 text-center"
     style="background: {{ theme.bg }}; border: 2px solid {{ theme.border }};">
    <div class="text-6xl mb-3">{{ theme.emoji }}</div>
    <h1 class="text-4xl font-bold mb-1" style="color: {{ theme.accent }};">
        {{ result.emotion | title }}
    </h1>
    <p class="text-gray-300 text-lg">{{ result.mood_description }}</p>

    <!-- Confidence bar -->
    <div class="mt-4 bg-gray-800 rounded-full h-2 max-w-xs mx-auto">
        <div class="h-2 rounded-full" style="width: {{ (result.confidence * 100) | round | int }}%; background: {{ theme.accent }};"></div>
    </div>
    <p class="text-sm text-gray-400 mt-1">{{ (result.confidence * 100) | round | int }}% confidence</p>
</div>

<!-- Song Grid -->
<h2 class="text-2xl font-semibold mb-4">🎵 Your Playlist</h2>
<div class="grid grid-cols-1 sm:grid-cols-2 lg:grid-cols-4 gap-4 mb-8">
{% for song in result.songs %}
<div class="bg-gray-800 rounded-xl p-5 border border-gray-700 hover:border-indigo-500 transition">
    <p class="font-semibold text-white truncate">{{ song.title }}</p>
    <p class="text-gray-400 text-sm mb-1">{{ song.artist }}</p>
    <p class="text-xs text-gray-500 mb-3">{{ song.genre }}</p>
    <p class="text-xs text-gray-400 italic mb-4">{{ song.why }}</p>
    <div class="flex gap-2">
        {% if song.spotify_url %}
        <a href="{{ song.spotify_url }}" target="_blank"
           class="text-xs bg-green-800 hover:bg-green-700 text-green-200 px-3 py-1 rounded-full">Spotify</a>
        {% endif %}
        {% if song.yt_music_url %}
        <a href="{{ song.yt_music_url }}" target="_blank"
           class="text-xs bg-red-900 hover:bg-red-800 text-red-200 px-3 py-1 rounded-full">YT Music</a>
        {% endif %}
    </div>
</div>
{% endfor %}
</div>

<a href="/detect" class="text-indigo-400 hover:underline">← Try Another Mood</a>

</body></html>
""")

print("results_v2.html created!")

### ✏️ In-Class Exercise 6b — Render a Results Page

Enrich the mock result and render it with the template.

In [ ]:
!pip install jinja2 -q
from jinja2 import Environment, FileSystemLoader

env = Environment(loader=FileSystemLoader("templates"))
tmpl = env.get_template("results_v2.html")

# Enrich the mock result (adds spotify/yt links)
enrich(mock_result)
theme = get_theme(mock_result["emotion"])

html = tmpl.render(result=mock_result, theme=theme)

# Save to a file we can open
with open("preview_results.html", "w") as f:
    f.write(html)

print("Saved! Download preview_results.html and open in browser.")
print(f"First song Spotify link: {mock_result['songs'][0]['spotify_url']}")

### ✏️ In-Class Exercise 6c — Wire It Into FastAPI

In [ ]:
import nest_asyncio; nest_asyncio.apply()
import threading, uvicorn
from fastapi import FastAPI, Request
from fastapi.templating import Jinja2Templates

app4 = FastAPI()
t4 = Jinja2Templates(directory="templates")

# TODO: Rename results_v2.html to results.html and add a GET /results route
# that reads ?key from query params, retrieves from cache, enriches, and renders

@app4.get("/results")
def results(request: Request, key: str = ""):
    # TODO: Use cache_retrieve(key), enrich(result), get_theme(emotion)
    # Return a TemplateResponse for results_v2.html with result + theme context
    pass

print("Route defined — fill in the TODO then test it!")

---
## 🏠 After-Class Exercises

---
### After-Class A: Add Apple Music Links

In [ ]:
def apple_music_link(title: str, artist: str) -> str:
    # Apple Music search URL: https://music.apple.com/search?term=query
    # TODO: Build and return the Apple Music search URL
    pass

print(apple_music_link("Happy", "Pharrell Williams"))

### After-Class B: Expand EMOTION_THEMES

Add the following emotions: `anxious`, `melancholy`, `energetic`, `romantic`, `nostalgic`, `hopeful`  
Pick appropriate emoji and colour palettes for each.

In [ ]:
# TODO: Add 6 new emotions to EMOTION_THEMES
# Tip: use colorhunt.co for palette inspiration

### After-Class C (Challenge): Sort Songs by Genre

In [ ]:
# TODO: In the results template, group songs by genre
# Hint: in the route, pre-process songs into a dict: {genre: [songs]}
# Then in the template, loop over genres and their songs

---
## ✅ Session 6 Complete!

**You learned:**
- URL encoding with `urllib.parse.quote`
- Building free music search links (no API needed)
- The `enrich()` function pattern
- Dynamic emotion theming with inline hex styles
- Jinja2 loops and conditionals for song cards

**Next session:** Sessions and Supabase — we give MoodBeats memory!